# Séance 5 — Variables temporelles (corrigé)

**Durée pratique : 2 h** &nbsp;·&nbsp; Ateliers 5.1 à 5.3

## Ce que vous saurez faire à la fin

- parser des dates hétérogènes sans perdre de lignes silencieusement ;
- extraire des variables calendaires exploitables en analyse et en BI ;
- rééchantillonner une série et calculer des moyennes mobiles.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :


---
## Atelier 5.1 — Parser des dates hétérogènes (35 min)

### Partie guidée

In [4]:
raw_sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')
print(raw_sales['date_commande'].dropna().sample(10, random_state=1).tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/home/komlavi/Bureau/Traitement de donnees/data/raw/ventes_brutes.csv'

Cinq formats coexistent dans la même colonne. Le piège est réel : `05/03/2024` peut se lire
5 mars ou 3 mai selon la convention. Une erreur ici déplace des milliers de lignes de mois
sans qu'aucune exception ne soit levée.

In [ ]:
# format='mixed' laisse pandas inférer ligne par ligne ; dayfirst lève l'ambiguïté jj/mm
order_dates = pd.to_datetime(raw_sales['date_commande'],
                             format='mixed', dayfirst=True, errors='coerce')

n_missing_before = raw_sales['date_commande'].isna().sum()
n_missing_after = order_dates.isna().sum()

print('Valeurs initialement absentes  :', n_missing_before)
print('Valeurs absentes après parsing :', n_missing_after)
print('Échecs de conversion           :', n_missing_after - n_missing_before)
print('Plage couverte :', order_dates.min(), '->', order_dates.max())

**`errors='coerce'` transforme les échecs en `NaT` au lieu de lever une exception.** C'est
commode, mais dangereux si vous ne comptez pas les échecs juste après. Une conversion qui
détruit 8 % des dates sans le signaler est une bombe à retardement.

### Partie autonome

In [ ]:
# Q1. Écrire une fonction parse_dates(series) qui renvoie (series_converties, report)
#     où report est un dict : n_total, n_already_missing, n_failed, success_rate,
#     min_date, max_date. Levez une ValueError si le taux de réussite est inférieur
#     à un seuil passé en paramètre (défaut 0,95).

MISSING_TOKENS = {'', 'nan', 'nat', 'none', 'null', 'na', 'n/a', '-'}


def parse_dates(series, min_success_rate=0.95, dayfirst=True):
    """Convertit une série de dates hétérogènes et rend compte de la conversion.

    Une valeur « déjà manquante » (NaN, chaîne vide, 'null'...) n'est pas un échec :
    on ne la compte pas dans le dénominateur du taux de réussite. Seules les valeurs
    renseignées mais impossibles à interpréter sont des échecs.
    """
    as_text = series.astype('string').str.strip()
    already_missing = series.isna() | as_text.str.lower().isin(MISSING_TOKENS)

    # Piège : avec format='mixed', dayfirst=True s'applique AUSSI aux dates ISO
    # ('2024-03-05' deviendrait le 3 mai). On parse donc les formats « année en tête »
    # sans dayfirst, et seulement les autres avec.
    year_first = as_text.str.match(r'^\d{4}[-/.]', na=False)
    parts = [
        pd.to_datetime(series[year_first & ~already_missing],
                       format='mixed', dayfirst=False, errors='coerce'),
        pd.to_datetime(series[~year_first & ~already_missing],
                       format='mixed', dayfirst=dayfirst, errors='coerce'),
    ]
    converted = pd.to_datetime(pd.concat(parts).reindex(series.index))

    failed = converted.isna() & ~already_missing
    n_total = int(len(series))
    n_already_missing = int(already_missing.sum())
    n_to_convert = n_total - n_already_missing
    n_failed = int(failed.sum())
    success_rate = 1.0 if n_to_convert == 0 else 1 - n_failed / n_to_convert

    report = {
        'n_total': n_total,
        'n_already_missing': n_already_missing,
        'n_failed': n_failed,
        'success_rate': round(success_rate, 4),
        'min_date': converted.min(),
        'max_date': converted.max(),
        'failed_examples': series[failed].drop_duplicates().head(5).tolist(),
    }

    if success_rate < min_success_rate:
        raise ValueError(
            f'Taux de conversion {success_rate:.2%} < seuil {min_success_rate:.0%} '
            f'({n_failed} échecs). Exemples : {report["failed_examples"]}'
        )
    return converted, report


order_dates, report = parse_dates(raw_sales['date_commande'])
for key, value in report.items():
    print(f'{key:<18}: {value}')

**Points clés de la correction (Q1).**

- On sépare les valeurs **déjà absentes** des **échecs** : une case vide dans la source n'est
  pas une erreur de parsing. Mélanger les deux fausse le taux de réussite.
- `failed_examples` n'était pas demandé, mais c'est ce qui rend le rapport utile : quand la
  `ValueError` tombe, on voit tout de suite *quels* formats posent problème.
- La fonction ne fait rien de silencieux : soit elle passe et documente, soit elle bloque.
- **Piège dans la partie guidée.** Avec `format='mixed'`, `dayfirst=True` s'applique aussi aux
  dates ISO : `pd.to_datetime('2024-03-05', format='mixed', dayfirst=True)` renvoie le
  **3 mai**. Toutes les dates `AAAA-MM-JJ` dont le jour est ≤ 12 sont inversées, sans erreur.
  La correction sépare les formats « année en tête » (parsés sans `dayfirst`) des autres.
  Vérifiez : la plage `min_date -> max_date` doit être plus courte qu'avec la cellule guidée,
  qui « invente » des commandes dans des mois futurs.

---
## Atelier 5.2 — Variables calendaires (45 min)

### Partie guidée

In [ ]:
if dataset_exists('ventes_transformees'):
    sales = load_dataset('ventes_transformees')
else:
    sales = raw_sales.assign(date_commande=order_dates).dropna(subset=['date_commande'])

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')
sales = sales.dropna(subset=['date_commande'])

date_accessor = sales['date_commande'].dt
sales['annee'] = date_accessor.year
sales['mois'] = date_accessor.month
sales['jour_semaine'] = date_accessor.dayofweek          # 0 = lundi
sales['nom_jour'] = date_accessor.day_name()
sales['semaine_iso'] = date_accessor.isocalendar().week.astype(int)
sales['trimestre'] = date_accessor.quarter
sales['heure'] = date_accessor.hour
sales['est_weekend'] = sales['jour_semaine'] >= 5

sales[['date_commande', 'annee', 'mois', 'nom_jour', 'semaine_iso', 'est_weekend']].head()

**`isocalendar().week` plutôt que `.week`.** La semaine ISO gère correctement les jours de
fin décembre appartenant à la semaine 1 de l'année suivante. C'est exactement le genre de
détail qui produit des écarts inexplicables entre deux tableaux de bord.

In [ ]:
# Encodage cyclique : le mois 12 est adjacent au mois 1, ce qu'un entier ne dit pas
sales['mois_sin'] = np.sin(2 * np.pi * sales['mois'] / 12)
sales['mois_cos'] = np.cos(2 * np.pi * sales['mois'] / 12)

cyclic = sales.groupby('mois')[['mois_sin', 'mois_cos']].first().round(3)
print(cyclic)
print()
print('Distance entre décembre et janvier :',
      round(float(np.hypot(*(cyclic.loc[12] - cyclic.loc[1]))), 3))
print('Distance entre janvier et février  :',
      round(float(np.hypot(*(cyclic.loc[1] - cyclic.loc[2]))), 3))

### Partie autonome

In [ ]:
# Q2. Ajouter les variables suivantes :
#     - periode_journee : 'matin' (<12), 'apres-midi' (12-17), 'soir' (>=18) ;
#     - est_fin_de_mois : les trois derniers jours du mois ;
#     - jours_depuis_debut : nombre de jours écoulés depuis la première commande ;
#     - anciennete_client_jours : écart entre la date de commande et la date d'inscription
#       du client (nécessite une jointure avec clients.csv).

CLIENT_KEY = 'id_client'
SIGNUP_COL = 'date_inscription'

# --- periode_journee : intervalles fermés à droite -> (-1, 11] (11, 17] (17, 23]
sales['periode_journee'] = pd.cut(sales['heure'], bins=[-1, 11, 17, 23],
                                  labels=['matin', 'apres-midi', 'soir'])

# --- est_fin_de_mois : on calcule à partir du nombre de jours du mois,
#     pas avec « jour >= 28 » qui serait faux pour février comme pour les mois à 31 jours
order_day = sales['date_commande'].dt.day
days_in_month = sales['date_commande'].dt.days_in_month
sales['est_fin_de_mois'] = (days_in_month - order_day) < 3

# --- jours_depuis_debut
first_order = sales['date_commande'].min()
sales['jours_depuis_debut'] = (sales['date_commande'] - first_order).dt.days

# --- anciennete_client_jours : jointure avec clients.csv
clients = pd.read_csv(RAW_DIR / 'clients.csv')
signup_dates, signup_report = parse_dates(clients[SIGNUP_COL], min_success_rate=0.90)
clients = clients.assign(**{SIGNUP_COL: signup_dates})

# Une clé client dupliquée dans la table de référence multiplierait les ventes :
# on dédoublonne explicitement, puis validate='many_to_one' le garantit.
n_dup_clients = clients.duplicated(CLIENT_KEY).sum()
clients_ref = (clients.sort_values(SIGNUP_COL)
               .drop_duplicates(CLIENT_KEY, keep='first')[[CLIENT_KEY, SIGNUP_COL]])

n_rows_before = len(sales)
sales = sales.drop(columns=[SIGNUP_COL], errors='ignore').merge(
    clients_ref, on=CLIENT_KEY, how='left', validate='many_to_one')
assert len(sales) == n_rows_before, 'la jointure a changé le nombre de lignes'

sales['anciennete_client_jours'] = (sales['date_commande'] - sales[SIGNUP_COL]).dt.days

print('Clients dupliqués dans clients.csv     :', n_dup_clients)
print('Ventes sans client retrouvé             :', sales[SIGNUP_COL].isna().sum())
print('Ancienneté négative (commande < inscr.) :', (sales['anciennete_client_jours'] < 0).sum())
print()
print(sales['periode_journee'].value_counts(dropna=False))
print()
print(sales[['date_commande', 'periode_journee', 'est_fin_de_mois',
             'jours_depuis_debut', 'anciennete_client_jours']].head())

**Points de vigilance (Q2).**

- **Dates sans heure.** Les formats du type `2024-03-05` donnent une heure à `00:00`, donc
  `periode_journee = 'matin'` par défaut. Si une partie de la source n'a pas d'heure, ces
  lignes gonflent artificiellement le « matin ». À vérifier avec
  `(sales['date_commande'].dt.normalize() == sales['date_commande']).mean()` avant de conclure
  quoi que ce soit sur les horaires.
- **Ancienneté négative.** Une commande antérieure à l'inscription est une incohérence de
  données (inscription ré-enregistrée, erreur de saisie, inversion jj/mm). On la **compte** et on
  la signale ; on ne la corrige pas en silence.
- **Jointure.** `how='left'` garde toutes les ventes, `validate='many_to_one'` fait planter
  la cellule si la table clients contient des doublons, et l'`assert` vérifie que le nombre de
  lignes n'a pas bougé. Trois garde-fous, zéro ligne perdue ou dupliquée sans le savoir.

In [ ]:
# Q3. Le chiffre d'affaires est-il différent le week-end ? Comparez le montant moyen
#     par commande selon est_weekend, puis selon le nom du jour. Commentez.

AMOUNT_CANDIDATES = ['montant', 'montant_ttc', 'montant_total', 'chiffre_affaires', 'ca']


def to_number(values):
    """Convertit '1 234,50 €' ou '1234.5' en float ; ce qui échoue devient NaN."""
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(float)
    cleaned = (values.astype('string')
               .str.replace(r'[€\s ]', '', regex=True)
               .str.replace(',', '.', regex=False))
    return pd.to_numeric(cleaned, errors='coerce')


def order_amount(df):
    for col in AMOUNT_CANDIDATES:
        if col in df.columns:
            return to_number(df[col])
    if {'quantite', 'prix_unitaire'} <= set(df.columns):
        return to_number(df['quantite']) * to_number(df['prix_unitaire'])
    raise KeyError(f'aucune colonne de montant trouvée parmi {list(df.columns)}')


sales['montant_commande'] = order_amount(sales)
print('Montants non exploitables :', sales['montant_commande'].isna().sum())

# Moyenne ET médiane : quelques très grosses commandes suffisent à tirer une moyenne
weekend_stats = (sales.groupby('est_weekend')['montant_commande']
                 .agg(n_commandes='count', moyenne='mean', mediane='median', ca_total='sum')
                 .rename(index={False: 'semaine', True: 'week-end'}))
weekend_stats['ca_par_jour'] = weekend_stats['ca_total'] / weekend_stats.index.map(
    {'semaine': 5, 'week-end': 2})
print(weekend_stats.round(2))

DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_stats = (sales.groupby('nom_jour')['montant_commande']
             .agg(n_commandes='count', moyenne='mean', mediane='median')
             .reindex(DAY_ORDER))
print()
print(day_stats.round(2))

fig, ax = plt.subplots(figsize=(9, 3.5))
day_stats['moyenne'].plot.bar(ax=ax, color=['#4C72B0'] * 5 + ['#DD8452'] * 2)
ax.set_ylabel('montant moyen par commande')
ax.set_xlabel('')
ax.set_title('Panier moyen par jour de la semaine')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

NameError: name 'sales' is not defined

**Comment commenter (Q3).** Trois questions différentes se cachent derrière « le CA est-il
différent le week-end » — il faut dire laquelle on regarde :

1. **Le panier moyen** (montant moyen par commande) : comparez `moyenne` *et* `mediane`. Si la
   moyenne bouge mais pas la médiane, l'écart vient de quelques grosses commandes, pas d'un
   comportement général.
2. **Le volume** (`n_commandes`) : il y a 5 jours de semaine pour 2 jours de week-end. Comparer
   des totaux bruts n'a aucun sens ; c'est pourquoi `ca_par_jour` ramène le CA à une journée.
3. **Le CA total** = volume × panier : un week-end peut avoir moins de commandes mais un panier
   plus élevé.

Rédigez votre conclusion à partir de vos chiffres, par exemple : *« le panier moyen du
week-end est supérieur de X % à celui de la semaine, mais la médiane est quasi identique : l'écart
tient à quelques commandes exceptionnelles »*. Pour aller plus loin, un test de Mann-Whitney
(`scipy.stats.mannwhitneyu`) dit si l'écart est distinguable du bruit.

---
## Atelier 5.3 — Rééchantillonnage et fenêtres glissantes (40 min)

Le jeu `capteurs.csv` contient trois séries horaires sur dix-huit mois, avec une panne de
onze jours, un capteur bloqué sur une valeur constante et des horodatages dupliqués.

In [ ]:
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv', parse_dates=['horodatage'])

print(sensors.shape)
print(sensors['id_capteur'].value_counts())
print('Doublons (capteur, horodatage) :',
      sensors.duplicated(['id_capteur', 'horodatage']).sum())

In [ ]:
# Nettoyage minimal avant toute analyse temporelle
sensors = sensors.drop_duplicates(['id_capteur', 'horodatage'])
sensors['humidite_pct'] = sensors['humidite_pct'].replace(-999, np.nan)

sensor_a = (sensors[sensors['id_capteur'] == 'CAP-A']
            .set_index('horodatage')
            .sort_index())

daily_temperature = sensor_a['temperature_c'].resample('D').mean()
print(daily_temperature.head())
print('\nJours sans aucune mesure :', daily_temperature.isna().sum())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
daily_temperature.plot(ax=ax, alpha=0.35, label='moyenne journalière')
(daily_temperature.rolling(7, min_periods=4).mean()
 .plot(ax=ax, linewidth=2, label='moyenne mobile 7 j'))
ax.set_ylabel('température (°C)')
ax.legend()
plt.tight_layout()
plt.show()

`min_periods=4` autorise le calcul dès le quatrième point disponible, au lieu d'attendre
sept valeurs complètes. Sans cela, chaque trou de la série efface les six points suivants
de la courbe lissée.

### Partie autonome

In [ ]:
# Q4. Détecter automatiquement la panne du capteur CAP-B :
#     trouver les plages horaires sans aucune mesure de plus de 24 heures.
#     Indice : réindexer sur une plage horaire complète puis repérer les suites de NaN.


def find_outages(sensors, sensor_id, min_hours=24, freq='h'):
    """Plages consécutives sans aucune mesure, d'au moins min_hours heures.

    Renvoie un DataFrame : debut, fin (derniers créneaux vides), duree_heures.
    """
    readings = sensors.loc[sensors['id_capteur'] == sensor_id, 'horodatage']
    observed_slots = pd.DatetimeIndex(readings).floor(freq).unique()

    # Grille horaire complète entre la première et la dernière mesure
    full_grid = pd.date_range(observed_slots.min(), observed_slots.max(), freq=freq)
    is_missing = pd.Series(~full_grid.isin(observed_slots), index=full_grid)

    # Chaque changement d'état (présent <-> absent) démarre un nouveau bloc
    block_id = is_missing.ne(is_missing.shift()).cumsum()
    gaps = (is_missing[is_missing]
            .groupby(block_id[is_missing])
            .agg(debut=lambda s: s.index.min(),
                 fin=lambda s: s.index.max(),
                 duree_heures='size'))

    return (gaps[gaps['duree_heures'] > min_hours]
            .reset_index(drop=True)
            .assign(duree_jours=lambda d: (d['duree_heures'] / 24).round(1)))


outages_b = find_outages(sensors, 'CAP-B')
print(outages_b)

# Contrôle croisé : les capteurs sans panne annoncée ne doivent rien remonter
for sensor_id in sorted(sensors['id_capteur'].unique()):
    print(sensor_id, '->', len(find_outages(sensors, sensor_id)), 'panne(s) > 24 h')

**Pourquoi cette méthode (Q4).** La difficulté est qu'une panne est *une absence de lignes* :
rien dans le fichier ne la signale. La réindexation sur une grille horaire complète transforme
les lignes manquantes en `True` visibles, puis l'astuce `ne(shift()).cumsum()` numérote les
blocs consécutifs — c'est le motif standard pour repérer des « suites » en pandas, réutilisé
tel quel en Q5. On doit retrouver ici la panne d'environ onze jours annoncée dans l'énoncé.

`floor('h')` rend la fonction robuste aux horodatages qui ne tombent pas pile sur l'heure
(`10:00:03`, par exemple), fréquents avec les vrais capteurs.

In [ ]:
# Q5. Détecter le capteur bloqué : trouver la période où CAP-C renvoie une valeur
#     strictement constante pendant plus de 24 relevés consécutifs.
#     Un capteur bloqué ne produit pas de valeur manquante : c'est ce qui rend
#     cette anomalie difficile à repérer et coûteuse à laisser passer.


def find_stuck_periods(sensors, sensor_id, column, min_readings=24):
    """Suites de plus de min_readings relevés consécutifs strictement identiques."""
    series = (sensors.loc[sensors['id_capteur'] == sensor_id]
              .sort_values('horodatage')
              .set_index('horodatage')[column]
              .dropna())

    run_id = series.ne(series.shift()).cumsum()
    runs = (series.groupby(run_id)
            .agg(valeur='first', n_releves='size')
            .assign(debut=series.index.to_series().groupby(run_id.values).min().values,
                    fin=series.index.to_series().groupby(run_id.values).max().values))

    stuck = runs[runs['n_releves'] > min_readings].reset_index(drop=True)
    stuck.insert(0, 'colonne', column)
    return stuck[['colonne', 'valeur', 'debut', 'fin', 'n_releves']]


MEASURE_COLS = [c for c in sensors.select_dtypes('number').columns]
print('Mesures analysées :', MEASURE_COLS)

stuck_c = pd.concat([find_stuck_periods(sensors, 'CAP-C', col) for col in MEASURE_COLS],
                    ignore_index=True)
print(stuck_c)

# Contrôle croisé sur les trois capteurs
for sensor_id in sorted(sensors['id_capteur'].unique()):
    n_found = sum(len(find_stuck_periods(sensors, sensor_id, col)) for col in MEASURE_COLS)
    print(sensor_id, '->', n_found, 'période(s) bloquée(s)')

**Remarques (Q5).**

- On teste **toutes les mesures numériques** : l'énoncé dit que CAP-C est bloqué, pas sur quelle
  colonne. Le contrôle croisé sur les trois capteurs vérifie qu'on n'a pas de faux positifs.
- Le `dropna()` évite qu'une valeur manquante (le `-999` remplacé par `NaN`) casse une suite :
  sinon un seul trou au milieu d'un blocage le couperait en deux blocs trop courts.
- Égalité **stricte** : avec des flottants, un capteur sain ne renvoie quasiment jamais deux
  fois exactement la même valeur, d'où l'efficacité du critère. Sur des mesures arrondies (au
  degré près, par exemple), il faudrait relever le seuil ou tester la variance glissante :
  `series.rolling(24).std() == 0`.

In [ ]:
# Q6. Agréger les ventes par semaine puis par mois. Tracer la série mensuelle du
#     chiffre d'affaires et commenter la saisonnalité observée.

revenue = sales.set_index('date_commande').sort_index()['montant_commande']

weekly_revenue = revenue.resample('W-MON', label='left', closed='left').sum()   # semaines lundi -> dimanche
monthly_revenue = revenue.resample('MS').sum()                                  # MS : étiquette au 1er du mois

# Les mois de bord peuvent être incomplets : on les repère pour ne pas les lire comme une baisse
first_day, last_day = revenue.index.min(), revenue.index.max()
partial_months = []
if first_day.day != 1:
    partial_months.append(first_day.to_period('M'))
if last_day.day != last_day.days_in_month:
    partial_months.append(last_day.to_period('M'))
print('Mois incomplets (à ne pas interpréter) :', [str(p) for p in partial_months])

print('\nCA hebdomadaire (5 premières semaines) :')
print(weekly_revenue.head().round(2))
print('\nCA mensuel :')
print(monthly_revenue.round(2))

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

weekly_revenue.plot(ax=axes[0], alpha=0.4, label='CA hebdomadaire')
weekly_revenue.rolling(4, min_periods=2).mean().plot(ax=axes[0], linewidth=2,
                                                     label='moyenne mobile 4 semaines')
axes[0].set_ylabel('CA (€)')
axes[0].legend()
axes[0].set_title('Chiffre d\'affaires hebdomadaire')

monthly_revenue.plot(ax=axes[1], marker='o', linewidth=2)
for period in partial_months:
    axes[1].axvspan(period.start_time, period.end_time, color='grey', alpha=0.15)
axes[1].set_ylabel('CA (€)')
axes[1].set_title('Chiffre d\'affaires mensuel (zones grisées : mois incomplets)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

# Profil saisonnier : CA moyen par mois calendaire, toutes années confondues,
# exprimé en indice (100 = mois moyen) pour lire directement les écarts en %
complete_months = monthly_revenue[~monthly_revenue.index.to_period('M').isin(partial_months)]
seasonal_index = complete_months.groupby(complete_months.index.month).mean()
seasonal_index = (100 * seasonal_index / seasonal_index.mean()).round(1)
print('\nIndice saisonnier (100 = mois moyen) :')
print(seasonal_index.to_string())

**Commenter la saisonnalité (Q6).** Appuyez-vous sur l'indice saisonnier plutôt que sur une
impression visuelle, et structurez le commentaire en trois temps :

1. **Tendance** : le CA progresse-t-il, stagne-t-il ou recule-t-il sur la période (la moyenne
   mobile hebdomadaire le montre mieux que la série brute) ?
2. **Saisonnalité** : quels mois sont au-dessus de 100, lesquels en dessous, et de combien ?
   Exemple de formulation : *« décembre atteint un indice de 135 (+35 % par rapport à un mois
   moyen), tandis qu'août tombe à 80 : creux estival puis pic des fêtes. »*
3. **Limites** : avec moins de deux ans de données, chaque mois calendaire n'est observé qu'une
   ou deux fois, donc un pic peut être un événement ponctuel (promotion, incident) plutôt qu'une
   saisonnalité. Les mois incomplets sont exclus de l'indice pour ne pas fabriquer de faux creux.

**Choix techniques.** `'MS'` étiquette chaque mois au 1er (plus lisible que `'ME'`, fin de mois) ;
`'W-MON'` avec `label='left', closed='left'` produit des semaines du lundi au dimanche, cohérentes
avec la semaine ISO de l'atelier 5.2.

In [ ]:
# Livrable : jeu de ventes enrichi
save_dataset(sales, 'ventes_temporelles')

Les fonctions `parse_dates`, `find_outages` et `find_stuck_periods` sont reprises dans
`src/temporal.py` (livrable demandé) : en production on les importe avec
`from src.temporal import parse_dates, find_outages, find_stuck_periods` au lieu de les recopier.

---
## Livrable de la séance

- la fonction `parse_dates` avec son rapport de conversion, dans `src/` ;
- le jeu de ventes enrichi de ses variables temporelles ;
- les deux fonctions de détection d'anomalies de capteur (panne et blocage) ;
- un graphique commenté de la saisonnalité mensuelle.